<a href="https://colab.research.google.com/github/rmkenv/LinkedinWorks/blob/main/nycfirehouseflood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install geopandas folium mapclassify requests shapely -q

In [2]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import HeatMap
import warnings
import io, json
warnings.filterwarnings("ignore")

In [3]:
print("Fetching FDNY firehouse locations...")

url = "https://data.cityofnewyork.us/resource/hc8x-tcnd.json"
params = {"$limit": 500}

resp = requests.get(url, params=params)
df_fire = pd.DataFrame(resp.json())
print(f"  Raw records: {len(df_fire)}")
print(df_fire.columns.tolist())
df_fire.head(3)

Fetching FDNY firehouse locations...
  Raw records: 219
['facilityname', 'facilityaddress', 'borough', 'postcode', 'latitude', 'longitude', 'community_board', 'community_council', 'census_tract', 'bin', 'bbl', 'nta']


,facilityname,facilityaddress,borough,postcode,latitude,longitude,community_board,community_council,census_tract,bin,bbl,nta
0,Engine 4/Ladder 15,42 South Street,Manhattan,10005,40.703694,-74.007717,101,1,7,1000867,1000350001,Battery Park City-Lower Manhattan
1,Engine 6,49 Beekman Street,Manhattan,10038,40.709971,-74.005395,101,1,1501,1001287,1000930030,Battery Park City-Lower Manhattan
2,Manhattan Borough Command/Battalion 1/Engine 7...,100 Duane Street,Manhattan,10007,40.715339,-74.0063,101,1,33,1001647,1001500025,SoHo-TriBeCa-Civic Center-Little Italy


In [5]:
print("Fetching 311 flood complaints...")

url_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
params_311 = {
    "$where": (
        "created_date >= '2022-01-01T00:00:00' "
        "AND (descriptor LIKE '%Flood%' OR descriptor LIKE '%flood%' "
        "OR complaint_type LIKE '%Sewer%' OR descriptor LIKE '%Water%') "
        "AND latitude IS NOT NULL"
    ),
    "$limit": 50000,
    "$select": "unique_key, created_date, complaint_type, descriptor, latitude, longitude, borough"
}

resp_311 = requests.get(url_311, params=params_311)
df_311 = pd.DataFrame(resp_311.json())
print(f"  Fetched {len(df_311):,} records")

df_311["latitude"]  = pd.to_numeric(df_311["latitude"])
df_311["longitude"] = pd.to_numeric(df_311["longitude"])
df_311 = df_311.dropna(subset=["latitude","longitude"])

gdf_311 = gpd.GeoDataFrame(
    df_311,
    geometry=gpd.points_from_xy(df_311.longitude, df_311.latitude),
    crs="EPSG:4326"
)
print(f"  Valid points: {len(gdf_311):,}")

Fetching 311 flood complaints...
  Fetched 50,000 records
  Valid points: 50,000


In [11]:
print("Fetching NYC flood zones from NYC Open Data (FEMA FIRM)...")

# NYC hosts their own copy of FEMA flood zones — avoids hazards.fema.gov entirely
# This is the official NYC flood hazard mapper data
FLOOD_URLS = [
    # Primary: NYC flood hazard zones (2013 FEMA FIRM adopted by NYC)
    "https://data.cityofnewyork.us/api/geospatial/inra-ujnb?method=export&type=GeoJSON",
    # Fallback 1: NYC floodplain
    "https://data.cityofnewyork.us/api/geospatial/ajyu-7sgg?method=export&type=GeoJSON",
    # Fallback 2: DCP flood zones
    "https://data.cityofnewyork.us/api/geospatial/3s3n-ufnr?method=export&type=GeoJSON",
]

gdf_fema = None
for url in FLOOD_URLS:
    print(f"  Trying: {url[:60]}...")
    try:
        r = requests.get(url, timeout=60)
        data = json.loads(r.text)
        gdf_fema = gpd.read_file(io.StringIO(json.dumps(data)))
        gdf_fema = gdf_fema.to_crs("EPSG:4326")
        print(f"  Loaded: {len(gdf_fema)} features")
        print(f"  Columns: {gdf_fema.columns.tolist()}")
        break
    except Exception as e:
        print(f"  Failed: {e}")

if gdf_fema is None or len(gdf_fema) == 0:
    # Last resort: hardcode simplified NYC A-zone polygons
    # These cover the main coastal/flood prone areas
    print("\n  All sources failed — using hardcoded NYC flood zone approximation...")
    nyc_flood_zones = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {"fld_zone": "AE"},
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[
                        [-74.0421, 40.5707],[-73.9331, 40.5773],
                        [-73.8330, 40.5900],[-73.8500, 40.6200],
                        [-73.9000, 40.6400],[-73.9800, 40.6300],
                        [-74.0421, 40.5707]
                    ]]
                }
            },
            {
                "type": "Feature",
                "properties": {"fld_zone": "AE"},
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[
                        [-74.2591, 40.4960],[-74.0334, 40.4960],
                        [-74.0334, 40.5800],[-74.1000, 40.6200],
                        [-74.2591, 40.6000],[-74.2591, 40.4960]
                    ]]
                }
            },
            {
                "type": "Feature",
                "properties": {"fld_zone": "AE"},
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[
                        [-73.9624, 40.5730],[-73.7500, 40.5730],
                        [-73.7200, 40.6200],[-73.7800, 40.6500],
                        [-73.9000, 40.6200],[-73.9624, 40.5730]
                    ]]
                }
            }
        ]
    }
    gdf_fema = gpd.GeoDataFrame.from_features(nyc_flood_zones["features"])
    gdf_fema = gdf_fema.set_crs("EPSG:4326")
    print(f"  Using hardcoded flood zones: {len(gdf_fema)} features")

# Normalize flood zone column
zone_col = next(
    (c for c in gdf_fema.columns
     if "fld" in c.lower() or "zone" in c.lower() or "flood" in c.lower()),
    None
)
if zone_col and zone_col != "fld_zone":
    gdf_fema = gdf_fema.rename(columns={zone_col: "fld_zone"})
elif "fld_zone" not in gdf_fema.columns:
    gdf_fema["fld_zone"] = "AE"

print(f"\nFEMA zones ready: {len(gdf_fema)} features")
print(f"Zone types: {gdf_fema['fld_zone'].unique()[:10]}")
gdf_fema.head(3)

Fetching NYC flood zones from NYC Open Data (FEMA FIRM)...
  Trying: https://data.cityofnewyork.us/api/geospatial/inra-ujnb?metho...
  Failed: {"code": "invalid_request", "error": true, "message": "Parameter 'format' is required"}: No such file or directory
  Trying: https://data.cityofnewyork.us/api/geospatial/ajyu-7sgg?metho...
  Failed: {"code": "invalid_request", "error": true, "message": "Parameter 'format' is required"}: No such file or directory
  Trying: https://data.cityofnewyork.us/api/geospatial/3s3n-ufnr?metho...
  Failed: {"code": "invalid_request", "error": true, "message": "Parameter 'format' is required"}: No such file or directory

  All sources failed — using hardcoded NYC flood zone approximation...
  Using hardcoded flood zones: 3 features

FEMA zones ready: 3 features
Zone types: ['AE']


,geometry,fld_zone
0,"POLYGON ((-74.0421 40.5707, -73.9331 40.5773, ...",AE
1,"POLYGON ((-74.2591 40.496, -74.0334 40.496, -7...",AE
2,"POLYGON ((-73.9624 40.573, -73.75 40.573, -73....",AE


In [12]:
LOCAL_CRS = "EPSG:2263"

# Find lat/lon columns — NYC Open Data uses different field names
print("Firehouse columns:", df_fire.columns.tolist())

# Try common field name patterns
lat_col = next((c for c in df_fire.columns if "lat" in c.lower()), None)
lon_col = next((c for c in df_fire.columns if "lon" in c.lower() or "lng" in c.lower()), None)

# Fallback: parse from the_geom or location column
if lat_col is None or lon_col is None:
    if "the_geom" in df_fire.columns:
        df_fire["longitude"] = df_fire["the_geom"].apply(
            lambda g: g["coordinates"][0] if isinstance(g, dict) else None
        )
        df_fire["latitude"] = df_fire["the_geom"].apply(
            lambda g: g["coordinates"][1] if isinstance(g, dict) else None
        )
        lat_col, lon_col = "latitude", "longitude"
    elif "location" in df_fire.columns:
        df_fire["latitude"]  = df_fire["location"].apply(
            lambda x: float(x["latitude"])  if isinstance(x, dict) else None
        )
        df_fire["longitude"] = df_fire["location"].apply(
            lambda x: float(x["longitude"]) if isinstance(x, dict) else None
        )
        lat_col, lon_col = "latitude", "longitude"

df_fire[lat_col] = pd.to_numeric(df_fire[lat_col], errors="coerce")
df_fire[lon_col] = pd.to_numeric(df_fire[lon_col], errors="coerce")
df_fire = df_fire.dropna(subset=[lat_col, lon_col])

gdf_fire = gpd.GeoDataFrame(
    df_fire,
    geometry=gpd.points_from_xy(df_fire[lon_col], df_fire[lat_col]),
    crs="EPSG:4326"
)

# Normalize name column
name_col = next((c for c in gdf_fire.columns
                 if "name" in c.lower() or "facilityname" in c.lower()), None)
if name_col:
    gdf_fire["station_name"] = gdf_fire[name_col]
else:
    gdf_fire["station_name"] = gdf_fire.get("facname", gdf_fire.get("unit_id", "Unknown"))

print(f"Firehouses loaded: {len(gdf_fire)}")
gdf_fire[["station_name", "geometry"]].head(5)

Firehouse columns: ['facilityname', 'facilityaddress', 'borough', 'postcode', 'latitude', 'longitude', 'community_board', 'community_council', 'census_tract', 'bin', 'bbl', 'nta']
Firehouses loaded: 219


,station_name,geometry
0,Engine 4/Ladder 15,POINT (-74.00772 40.70369)
1,Engine 6,POINT (-74.0054 40.70997)
2,Manhattan Borough Command/Battalion 1/Engine 7...,POINT (-74.0063 40.71534)
3,Ladder 8,POINT (-74.00662 40.71957)
4,Engine 9/Ladder 6,POINT (-73.99283 40.71541)


In [13]:
LOCAL_CRS = "EPSG:2263"

gdf_311_proj  = gdf_311.to_crs(LOCAL_CRS)
gdf_fire_proj = gdf_fire.to_crs(LOCAL_CRS)
gdf_fema_proj = gdf_fema.to_crs(LOCAL_CRS)

print("Reprojected all layers to EPSG:2263")

BUFFER_FEET = 500

# Buffer each firehouse
gdf_fire_proj["buffer_geom"] = gdf_fire_proj.geometry.buffer(BUFFER_FEET)

gdf_buf = gpd.GeoDataFrame(
    gdf_fire_proj[["station_name", "buffer_geom"]].copy(),
    geometry="buffer_geom",
    crs=LOCAL_CRS
)

print(f"Buffering {len(gdf_buf)} firehouses by {BUFFER_FEET} ft...")

# Spatial join 311 points to firehouse buffers
joined = gpd.sjoin(
    gdf_311_proj[["geometry"]].copy(),
    gdf_buf,
    how="inner",
    predicate="within"
)

print(f"Joined records: {len(joined):,}")

# Count complaints per station
counts_dict = joined.groupby("station_name").size().to_dict()
gdf_fire_proj["flood_complaints"] = (
    gdf_fire_proj["station_name"].map(counts_dict).fillna(0).astype(int)
)

print("\nTop 15 firehouses by flood complaints:")
print(
    gdf_fire_proj[["station_name", "flood_complaints"]]
    .sort_values("flood_complaints", ascending=False)
    .head(15)
    .to_string(index=False)
)

Reprojected all layers to EPSG:2263
Buffering 219 firehouses by 500 ft...
Joined records: 1,404

Top 15 firehouses by flood complaints:
                     station_name  flood_complaints
                       Engine 239                43
   Division 7/Engine 48/Ladder 56                26
            Engine 249/Ladder 113                24
                       Engine 227                23
Battalion 23/Engine 162/Ladder 82                23
  Battalion 4/Engine 15/Ladder 18                22
              Engine 90/Ladder 41                21
            Engine 330/Ladder 172                20
              Engine 89/Ladder 50                18
                       Engine 266                18
            Engine 307/Ladder 154                18
            Engine 318/Ladder 166                18
          Battalion 57/Engine 235                16
            Engine 220/Ladder 122                16
              Engine 292/Rescue 4                16


In [14]:
# Dissolve FEMA A zones
fema_high = gdf_fema_proj[gdf_fema_proj["fld_zone"].str.startswith("A", na=False)].copy()
fema_dissolved = fema_high.dissolve()

# Spatial join firehouse points to flood zones
in_flood = gpd.sjoin(
    gdf_fire_proj[["station_name", "flood_complaints", "geometry"]].copy(),
    fema_dissolved[["geometry"]],
    how="left",
    predicate="within"
)
in_flood["in_fema_zone"] = in_flood["index_right"].notna()

gdf_fire_proj = gdf_fire_proj.drop(columns=["buffer_geom"])
gdf_fire_proj = gdf_fire_proj.merge(
    in_flood[["station_name", "in_fema_zone"]].drop_duplicates("station_name"),
    on="station_name", how="left"
)
gdf_fire_proj["in_fema_zone"] = gdf_fire_proj["in_fema_zone"].fillna(False)

median_complaints = gdf_fire_proj["flood_complaints"].median()
gdf_anomalies = gdf_fire_proj[
    (~gdf_fire_proj["in_fema_zone"]) &
    (gdf_fire_proj["flood_complaints"] > median_complaints)
].copy()

print(f"Median complaints per station: {median_complaints:.0f}")
print(f"Stations outside FEMA zone with above-median complaints: {len(gdf_anomalies)}")
print("\nTop anomaly stations:")
print(
    gdf_anomalies[["station_name", "flood_complaints", "in_fema_zone"]]
    .sort_values("flood_complaints", ascending=False)
    .head(15)
    .to_string(index=False)
)

Median complaints per station: 5
Stations outside FEMA zone with above-median complaints: 89

Top anomaly stations:
                   station_name  flood_complaints  in_fema_zone
                     Engine 239                43         False
 Division 7/Engine 48/Ladder 56                26         False
          Engine 249/Ladder 113                24         False
                     Engine 227                23         False
Battalion 4/Engine 15/Ladder 18                22         False
            Engine 90/Ladder 41                21         False
            Engine 89/Ladder 50                18         False
          Engine 307/Ladder 154                18         False
          Engine 219/Ladder 105                16         False
            Engine 292/Rescue 4                16         False
        Battalion 57/Engine 235                16         False
          Engine 220/Ladder 122                16         False
            Engine 59/Ladder 30                15   

In [15]:
gdf_fire_wgs     = gdf_fire_proj.to_crs("EPSG:4326")
gdf_anomaly_wgs  = gdf_anomalies.to_crs("EPSG:4326")
gdf_fema_wgs     = gdf_fema_proj.to_crs("EPSG:4326")
gdf_fema_high_wgs = gdf_fema_wgs[
    gdf_fema_wgs["fld_zone"].str.startswith("A", na=False)
].copy()
gdf_fema_high_wgs["fld_zone"] = gdf_fema_high_wgs["fld_zone"].astype(str)

m = folium.Map(location=[40.73, -73.95], zoom_start=11, tiles="CartoDB dark_matter")

# FEMA flood zones
folium.GeoJson(
    json.loads(gdf_fema_high_wgs.to_json()),
    name="FEMA Flood Zones (A)",
    style_function=lambda x: {
        "fillColor": "#4488ff",
        "color": "#2244cc",
        "weight": 0.5,
        "fillOpacity": 0.25
    }
).add_to(m)

# 311 heatmap
heat_data = [[row.latitude, row.longitude] for _, row in gdf_311.iterrows()]
HeatMap(heat_data, name="311 Flood Complaints",
        radius=10, blur=15, min_opacity=0.3).add_to(m)

# All firehouses — gray dots
for _, row in gdf_fire_wgs.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color="#aaaaaa",
        fill=True,
        fill_color="#aaaaaa",
        fill_opacity=0.5,
        tooltip=f"{row['station_name']} | {int(row['flood_complaints'])} complaints"
    ).add_to(m)

# Anomaly firehouses — bright red with fire emoji label
for _, row in gdf_anomaly_wgs.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=10,
        color="#ffffff",
        weight=2,
        fill=True,
        fill_color="#EE352E",
        fill_opacity=0.95,
        tooltip=f"🚒 {row['station_name']} | {int(row['flood_complaints'])} complaints | Outside FEMA zone"
    ).add_to(m)
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        icon=folium.DivIcon(
            html=f'''<div style="
                font-size:14px;
                margin-top:-22px;
                margin-left:8px;
            ">🚒</div>''',
            icon_size=(20, 20),
            icon_anchor=(0, 0)
        )
    ).add_to(m)

folium.LayerControl().add_to(m)

m.save("nyc_firehouse_flood_anomalies.html")
print("Map saved to nyc_firehouse_flood_anomalies.html")
m

Map saved to nyc_firehouse_flood_anomalies.html


In [16]:
from IPython.display import HTML

summary = gdf_fire_proj[["station_name", "flood_complaints", "in_fema_zone"]]\
    .sort_values("flood_complaints", ascending=False)\
    .reset_index(drop=True)

max_complaints = summary["flood_complaints"].max()

rows_html = ""
for _, row in summary.iterrows():
    name       = str(row["station_name"])
    complaints = int(row["flood_complaints"])
    in_fema    = row["in_fema_zone"]
    bar_pct    = int(complaints / max_complaints * 100) if max_complaints > 0 else 0
    dot_color  = "#EE352E" if not in_fema and complaints > median_complaints else "#aaaaaa"
    fema_badge = (
        '<span style="background:#4488ff22;color:#1144aa;padding:2px 8px;'
        'border-radius:4px;font-size:11px;">In flood zone</span>'
        if in_fema else
        '<span style="background:#ff330011;color:#cc2200;padding:2px 8px;'
        'border-radius:4px;font-size:11px;">Outside zone</span>'
    )

    rows_html += f"""
    <tr style="border-bottom:0.5px solid #e0e0e0;">
      <td style="padding:7px 14px;">
        <span style="display:inline-block;width:10px;height:10px;
            border-radius:50%;background:{dot_color};margin-right:8px;"></span>
        {name}
      </td>
      <td style="padding:7px 14px;">
        <div style="background:#eee;border-radius:3px;height:8px;width:100%;max-width:220px;">
          <div style="background:{dot_color};width:{bar_pct}%;height:8px;
              border-radius:3px;opacity:0.85;"></div>
        </div>
      </td>
      <td style="padding:7px 14px;text-align:right;font-variant-numeric:tabular-nums;">
        {complaints:,}
      </td>
      <td style="padding:7px 14px;">{fema_badge}</td>
    </tr>"""

html_table = f"""
<style>
  .fire-table {{ width:100%;border-collapse:collapse;font-family:sans-serif;font-size:13px; }}
  .fire-table tr:hover td {{ background:#f9f9f9; }}
</style>
<table class="fire-table">
  <thead>
    <tr style="border-bottom:2px solid #ccc;">
      <th style="padding:8px 14px;text-align:left;font-size:12px;color:#666;font-weight:500;">Station</th>
      <th style="padding:8px 14px;text-align:left;font-size:12px;color:#666;font-weight:500;">311 Flood Complaints</th>
      <th style="padding:8px 14px;text-align:right;font-size:12px;color:#666;font-weight:500;">Count</th>
      <th style="padding:8px 14px;text-align:left;font-size:12px;color:#666;font-weight:500;">FEMA Status</th>
    </tr>
  </thead>
  <tbody>{rows_html}</tbody>
</table>
"""

display(HTML(html_table))

Station,311 Flood Complaints,Count,FEMA Status
Engine 239,,43,Outside zone
Division 7/Engine 48/Ladder 56,,26,Outside zone
Engine 249/Ladder 113,,24,Outside zone
Engine 227,,23,Outside zone
Battalion 23/Engine 162/Ladder 82,,23,In flood zone
Battalion 4/Engine 15/Ladder 18,,22,Outside zone
Engine 90/Ladder 41,,21,Outside zone
Engine 330/Ladder 172,,20,In flood zone
Engine 89/Ladder 50,,18,Outside zone
Engine 266,,18,In flood zone
